# Reddit Data Analysis Pipeline (PySpark)

This notebook implements a Big Data processing pipeline for Reddit XML data using **Apache Spark (PySpark)**.

**Key Features:**
- **Full Spark Pipeline**: No external scripts required.
- **In-Memory Unzip**: Reads ZIP files using Python `glob` + `zipfile` and distributes processing via Spark.
- **Spark-Native Cleaning**: Uses Spark SQL functions (`regexp_replace`, `trim`) for high-performance text processing.
- **Label Integration**: Merges user labels from external truth file.
- **Compatibility**: Configured to run on Windows with Java 8/11.

## 1. Environment Setup

In [ ]:
import os
import sys

# Force usage of Java 8 (JRE 1.8) to handle Spark compatibility on Windows
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jre-1.8"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Ensure PySpark workers use the same Python executable as the driver
# Use Short Path (8.3) to avoid space issues in Spark launcher on Windows
# "Progetto big data" -> "PROGET~1"
SAFE_PYTHON_PATH = r"C:\Users\Franco\Desktop\PROGET~1\.venv\Scripts\python.exe"

os.environ['PYSPARK_PYTHON'] = SAFE_PYTHON_PATH
os.environ['PYSPARK_DRIVER_PYTHON'] = SAFE_PYTHON_PATH

print("Environment configured.")

## 2. Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, trim, length, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder \
    .appName("RedditDataAnalysis") \
    .master("local[*]") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

print("Spark Session Created.")
print(f"Spark Version: {spark.version}")

## 3. Data Ingestion (Zip Reading & Parsing)
We use `glob` to list files and `sc.parallelize` to distribute the work. This bypasses Hadoop's path handling issues on Windows.

In [ ]:
import glob
import zipfile
import re
import xml.etree.ElementTree as ET

# Path to ZIP files
# Notebook is now in '01 data', so data is in 'data' subdir relative to notebook
BASE_DIR = os.getcwd()
ZIP_PATTERN = os.path.join(BASE_DIR, "data", "*.zip")

print(f"Reading ZIPs from: {ZIP_PATTERN}")

files = glob.glob(ZIP_PATTERN)
print(f"Found {len(files)} ZIP files.")

def parse_zip_path(filepath):
    """
    Args: filepath (local absolute path)
    Returns: List of rows
    """
    rows = []
    try:
        chunk_match = re.search(r'chunk(\d+)', filepath, re.IGNORECASE)
        chunk_id = chunk_match.group(1) if chunk_match else "Unknown"

        with zipfile.ZipFile(filepath, "r") as z:
            xml_files = [f for f in z.namelist() if f.lower().endswith('.xml')]
            
            for xml_filename in xml_files:
                try:
                    with z.open(xml_filename) as f:
                        xml_content = f.read()
                    
                    root = ET.fromstring(xml_content)
                    
                    sub_node = root.find('ID')
                    raw_id = sub_node.text.strip() if sub_node is not None and sub_node.text else None
                    subject_id = raw_id.replace("subject", "") if raw_id else None

                    for writing in root.findall('WRITING'):
                        title = writing.find('TITLE').text or ""
                        date = writing.find('DATE').text or ""
                        info = writing.find('INFO').text or ""
                        text = writing.find('TEXT').text or ""
                        
                        rows.append((subject_id, chunk_id, date.strip(), title.strip(), info.strip(), text.strip()))
                except:
                    continue
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return []
    return rows

# Parallelize and Parse
files_rdd = spark.sparkContext.parallelize(files, numSlices=8)
rows_rdd = files_rdd.flatMap(parse_zip_path)

# Define Schema
schema = StructType([
    StructField("Subject ID", StringType(), True),
    StructField("Chunk", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Title_Raw", StringType(), True),
    StructField("Info", StringType(), True),
    StructField("Text_Raw", StringType(), True)
])

df_raw = spark.createDataFrame(rows_rdd, schema)
print("Raw DataFrame Created.")
df_raw.show(5)

## 4. Spark-Native Preprocessing

In [ ]:
# Pre-compile regex patterns for Spark

# 1. Newlines
df_clean = df_raw.withColumn("Title", regexp_replace(col("Title_Raw"), r"[\r\n]+", " ")) \
                 .withColumn("Text", regexp_replace(col("Text_Raw"), r"[\r\n]+", " "))

# 2. Markdown Links [Label](URL) -> Label
df_clean = df_clean.withColumn("Title", regexp_replace(col("Title"), r"\[([^]]+)\]\([^)]+\)", "$1")) \
                   .withColumn("Text", regexp_replace(col("Text"), r"\[([^]]+)\]\([^)]+\)", "$1"))

# 3. Raw URLs
df_clean = df_clean.withColumn("Title", regexp_replace(col("Title"), r"http\S+", "")) \
                   .withColumn("Text", regexp_replace(col("Text"), r"http\S+", ""))

# 4. Trim
df_clean = df_clean.withColumn("Title", trim(regexp_replace(col("Title"), r"\s+", " "))) \
                   .withColumn("Text", trim(regexp_replace(col("Text"), r"\s+", " ")))

# 5. Filter Empty Text
df_filtered = df_clean.filter(length(col("Text")) > 0)

## 5. Label User Integration
Reading `risk-golden-truth-test.txt` and joining it with the main dataset.

In [ ]:
# Path to labels file
# Note: Running from '01 data', so file is in 'data' folder
LABEL_FILE = os.path.join(BASE_DIR, "data", "risk-golden-truth-test.txt")

print(f"Reading labels from: {LABEL_FILE}")

# Read Text File
df_labels_raw = spark.read.option("delimiter", "\t").csv(LABEL_FILE)

# Rename columns and clean ID
# Col _c0 is Subject (e.g. subject3307), _c1 is Label (e.g. 1)
df_labels = df_labels_raw.select(
    regexp_replace(col("_c0"), "subject", "").alias("Subject ID"),
    col("_c1").alias("Label_User")
)

print("Labels loaded.")
df_labels.show(5)

# Join with Main DataFrame
# Left Join to keep all rows in the dataset even if label is missing (though it shouldn't be)
df_joined = df_filtered.join(df_labels, "Subject ID", "left")

## 6. Sorting and Export

In [ ]:
# Sort
df_sorted = df_joined.withColumn("Subject_ID_Num", col("Subject ID").cast("int")) \
                     .orderBy(col("Subject_ID_Num").asc(), col("Date").asc())

# Select final columns
final_df = df_sorted.select("Subject ID", "Chunk", "Date", "Title", "Info", "Text", "Label_User")

print(f"Final Row Count: {final_df.count()}")
final_df.show(5)

# export to 01 data folder (Current Dir)
output_path = os.path.join(BASE_DIR, "reddit_dataset.csv")

print("Collecting data to driver for export...")
pdf = final_df.toPandas()

import csv
if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))

pdf.to_csv(output_path, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"Saved to {output_path}")

In [ ]:
spark.stop()